From Page 20: A new distributed architecture for evaluating AI-based security systems at the edge: Network TON_IoT datasets


In [1]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
import category_encoders as ce

DATA_FOLDER ='/mnt/d/TON_IoT_Dataset'


In [2]:
# Make smaller dataset for development purposes

# FRACTION = 0.2

# label_df = pd.read_csv(Path(DATA_FOLDER) / 'combined-labels.csv')

# sampled_df = label_df.groupby(['label', 'type'], group_keys=False).apply(lambda g: g.sample(frac=FRACTION, random_state=42))
# sampled_df = sampled_df.reset_index(drop=True)
# sampled_df.sort_values(by=['time_group'], inplace=True)
# sampled_df.to_csv(Path(DATA_FOLDER) / 'small' / 'combined-labels.csv', index=False)
# sampled_df

In [3]:
# Perform sampling on the data files to make smaller dataset
# CHUNK_SIZE = 1000000

# def process_chunk(chunk):
#     chunk = chunk[chunk['time_group'].isin(sampled_df['time_group'])]
#     return chunk

# chunks = []

# for i, chunk in enumerate(pd.read_csv(Path(DATA_FOLDER) / 'combined.csv', chunksize=CHUNK_SIZE)):
#     print(f"Processing chunk {i+1} (rows {i*CHUNK_SIZE} to {(i+1)*CHUNK_SIZE - 1})...")
#     chunk = process_chunk(chunk)
#     chunks.append(chunk)

# df = pd.concat(chunks, ignore_index=True)
# df.to_csv(Path(DATA_FOLDER) / 'small' / 'combined.csv', index=False)
# df.head(1000)

In [4]:
raw_df = pd.read_csv(Path(DATA_FOLDER) / 'small' / 'combined.csv')

raw_df['src'] = raw_df['src_ip'] + ':' + raw_df['src_port'].astype('str')
raw_df['dst'] = raw_df['dst_ip'] + ':' + raw_df['dst_port'].astype('str')

# Reorder columns to have 'src' and 'dst' as the second and third columns
cols = raw_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('dst')))
cols.insert(1, cols.pop(cols.index('src')))
raw_df = raw_df[cols]

# Remove unnecessary columns
raw_df = raw_df.drop(columns=['uid', 'ts','src_ip', 'dst_ip', 'src_port','dst_port','http_uri','weird_name','weird_addl','weird_notice','dns_query','ssl_subject','ssl_issuer','http_user_agent','label'])
raw_df = raw_df.rename(columns={"type": "label"})

# Fix mislabeled values in 'src_bytes' column
raw_df.loc[raw_df['src_bytes'] == '0.0.0.0', 'src_bytes'] = 0
raw_df['src_bytes'] = raw_df['src_bytes'].astype('int')

raw_df.head(100)


/tmp/ipykernel_25685/2183621750.py:1: DtypeWarning: Columns (8,46) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(Path(DATA_FOLDER) / 'small' / 'combined.csv')


,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_method,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group
0,192.168.1.152:1880,192.168.1.152:51782,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,8
1,192.168.1.152:34296,192.168.1.152:10502,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,8
2,192.168.1.79:39306,192.168.1.255:15600,udp,-,0.000000,0,0,S0,0,1,...,-,-,-,0,0,0,-,-,normal,8
3,192.168.1.133:5353,224.0.0.251:5353,udp,dns,0.000000,0,0,S0,0,1,...,-,-,-,0,0,0,-,-,normal,8
4,192.168.1.152:1880,192.168.1.152:51782,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,192.168.1.152:1880,192.168.1.152:51782,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,41
96,192.168.1.79:53147,192.168.1.255:15600,udp,-,0.000000,0,0,S0,0,1,...,-,-,-,0,0,0,-,-,normal,41
97,192.168.1.152:34296,192.168.1.152:10502,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,normal,41
98,192.168.1.133:5353,224.0.0.251:5353,udp,dns,3.991139,351,0,S0,0,3,...,-,-,-,0,0,0,-,-,normal,41


In [5]:
le = LabelEncoder()
le.fit(raw_df.label.values)
raw_df['label'] = le.transform(raw_df['label'])
raw_df

,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_method,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group
0,192.168.1.152:1880,192.168.1.152:51782,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,5,8
1,192.168.1.152:34296,192.168.1.152:10502,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,5,8
2,192.168.1.79:39306,192.168.1.255:15600,udp,-,0.000000,0,0,S0,0,1,...,-,-,-,0,0,0,-,-,5,8
3,192.168.1.133:5353,224.0.0.251:5353,udp,dns,0.000000,0,0,S0,0,1,...,-,-,-,0,0,0,-,-,5,8
4,192.168.1.152:1880,192.168.1.152:51782,tcp,-,0.000000,0,0,OTH,0,0,...,-,-,-,0,0,0,-,-,5,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4247188,192.168.1.30:3050,192.168.1.194:3050,tcp,-,0.000011,0,0,REJ,0,1,...,-,-,-,0,0,0,-,-,2,97337
4247189,192.168.1.30:3050,192.168.1.184:3050,tcp,-,0.000016,0,0,REJ,0,1,...,-,-,-,0,0,0,-,-,2,97337
4247190,192.168.1.30:3050,192.168.1.184:3050,tcp,-,0.000113,0,0,REJ,0,1,...,-,-,-,0,0,0,-,-,2,97337
4247191,192.168.1.30:3050,192.168.1.194:3050,tcp,-,0.000088,0,0,REJ,0,1,...,-,-,-,0,0,0,-,-,2,97337


In [6]:
le.classes_

array(['backdoor', 'ddos', 'dos', 'injection', 'mitm', 'normal',
       'password', 'ransomware', 'scanning', 'xss'], dtype=object)

In [7]:
label_df = pd.read_csv(Path(DATA_FOLDER) / 'small' / 'combined-labels.csv')
train_idx = label_df[label_df['type'] == 'train']['time_group'].values
test_idx = label_df[label_df['type'] == 'test']['time_group'].values

train_df = raw_df[raw_df['time_group'].isin(train_idx)]
test_df = raw_df[raw_df['time_group'].isin(test_idx)]

print(train_df.shape[0] + test_df.shape[0])

4247193


In [8]:
categorical_cols = ['proto','service','conn_state','dns_qclass','dns_qtype','dns_rcode','dns_AA','dns_RD','dns_RA','dns_rejected','ssl_version','ssl_cipher','ssl_resumed','http_referrer','ssl_established','http_method','http_version','http_status_code','http_orig_mime_types','http_resp_mime_types','http_trans_depth']
target_encoder = ce.TargetEncoder(cols=categorical_cols)
target_encoder.fit(train_df, train_df.label)
train_df = target_encoder.transform(train_df)
train_df

,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_method,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group
0,192.168.1.152:1880,192.168.1.152:51782,4.437305,4.103382,0.000000,0,0,5.797699,0,0,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,8
1,192.168.1.152:34296,192.168.1.152:10502,4.437305,4.103382,0.000000,0,0,5.797699,0,0,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,8
2,192.168.1.79:39306,192.168.1.255:15600,4.459104,4.103382,0.000000,0,0,7.591863,0,1,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,8
3,192.168.1.133:5353,224.0.0.251:5353,4.459104,4.422711,0.000000,0,0,7.591863,0,1,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,8
4,192.168.1.152:1880,192.168.1.152:51782,4.437305,4.103382,0.000000,0,0,5.797699,0,0,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4247188,192.168.1.30:3050,192.168.1.194:3050,4.437305,4.103382,0.000011,0,0,3.513345,0,1,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,2,97337
4247189,192.168.1.30:3050,192.168.1.184:3050,4.437305,4.103382,0.000016,0,0,3.513345,0,1,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,2,97337
4247190,192.168.1.30:3050,192.168.1.184:3050,4.437305,4.103382,0.000113,0,0,3.513345,0,1,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,2,97337
4247191,192.168.1.30:3050,192.168.1.194:3050,4.437305,4.103382,0.000088,0,0,3.513345,0,1,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,2,97337


In [9]:
scaler = StandardScaler()
cols_to_norm = list(set(list(train_df.columns)) - set(list(['src', 'dst', 'label', 'time_group'])))
train_df[cols_to_norm] = scaler.fit_transform(train_df[cols_to_norm])

train_df['features'] = train_df[cols_to_norm].values.tolist()
train_df

,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group,features
0,192.168.1.152:1880,192.168.1.152:51782,-0.066135,-0.364823,-0.090490,-0.025867,-0.022454,0.563426,-0.00697,-0.011315,...,-0.00054,0.035948,-0.006736,-0.000758,0.029674,-0.008269,-0.016017,5,8,"[-0.0007576968612438045, -0.008346005540407472..."
1,192.168.1.152:34296,192.168.1.152:10502,-0.066135,-0.364823,-0.090490,-0.025867,-0.022454,0.563426,-0.00697,-0.011315,...,-0.00054,0.035948,-0.006736,-0.000758,0.029674,-0.008269,-0.016017,5,8,"[-0.0007576968612438045, -0.008346005540407472..."
2,192.168.1.79:39306,192.168.1.255:15600,0.387707,-0.364823,-0.090490,-0.025867,-0.022454,1.308243,-0.00697,-0.008779,...,-0.00054,0.035948,-0.006736,-0.000758,0.029674,-0.008269,-0.016017,5,8,"[-0.0007576968612438045, -0.008346005540407472..."
3,192.168.1.133:5353,224.0.0.251:5353,0.387707,-0.019209,-0.090490,-0.025867,-0.022454,1.308243,-0.00697,-0.008779,...,-0.00054,0.035948,-0.006736,-0.000758,0.029674,-0.008269,-0.016017,5,8,"[-0.0007576968612438045, -0.008346005540407472..."
4,192.168.1.152:1880,192.168.1.152:51782,-0.066135,-0.364823,-0.090490,-0.025867,-0.022454,0.563426,-0.00697,-0.011315,...,-0.00054,0.035948,-0.006736,-0.000758,0.029674,-0.008269,-0.016017,5,8,"[-0.0007576968612438045, -0.008346005540407472..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4247188,192.168.1.30:3050,192.168.1.194:3050,-0.066135,-0.364823,-0.090490,-0.025867,-0.022454,-0.384885,-0.00697,-0.008779,...,-0.00054,0.035948,-0.006736,-0.000758,0.029674,-0.008269,-0.016017,2,97337,"[-0.0007576968612438045, -0.005131093080033863..."
4247189,192.168.1.30:3050,192.168.1.184:3050,-0.066135,-0.364823,-0.090490,-0.025867,-0.022454,-0.384885,-0.00697,-0.008779,...,-0.00054,0.035948,-0.006736,-0.000758,0.029674,-0.008269,-0.016017,2,97337,"[-0.0007576968612438045, -0.005131093080033863..."
4247190,192.168.1.30:3050,192.168.1.184:3050,-0.066135,-0.364823,-0.090489,-0.025867,-0.022454,-0.384885,-0.00697,-0.008779,...,-0.00054,0.035948,-0.006736,-0.000758,0.029674,-0.008269,-0.016017,2,97337,"[-0.0007576968612438045, -0.005131093080033863..."
4247191,192.168.1.30:3050,192.168.1.194:3050,-0.066135,-0.364823,-0.090489,-0.025867,-0.022454,-0.384885,-0.00697,-0.008779,...,-0.00054,0.035948,-0.006736,-0.000758,0.029674,-0.008269,-0.016017,2,97337,"[-0.0007576968612438045, -0.005131093080033863..."


In [10]:
test_df = target_encoder.transform(test_df)
test_df

,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_method,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group
33795,127.0.0.1:42100,127.0.0.1:7878,4.437305,4.103382,0.000000,0,0,5.797699,0,0,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,7134
33796,192.168.1.190:43530,192.168.1.190:7878,4.437305,4.103382,0.000000,0,0,5.797699,0,0,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,7134
33797,192.168.1.152:50148,192.168.1.190:53,4.459104,4.422711,0.000433,0,298,1.371472,0,0,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,7134
33798,192.168.1.152:50148,192.168.1.190:53,4.459104,4.422711,0.000053,130,0,7.591863,0,2,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,7134
33799,192.168.1.152:50148,192.168.1.190:53,4.459104,4.422711,0.000053,130,0,7.591863,0,2,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,5,7134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3909020,192.168.1.31:33242,192.168.1.169:443,4.437305,1.573523,0.030442,983,1720,7.101582,0,7,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,8,94515
3909021,192.168.1.31:33200,192.168.1.169:443,4.437305,1.573523,0.030531,1012,1720,7.101582,0,7,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,8,94515
3909022,192.168.1.31:33108,192.168.1.169:443,4.437305,1.573523,0.031843,982,1720,7.101582,0,7,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,8,94515
3909023,192.168.1.31:36072,192.168.1.169:5800,4.437305,6.346691,0.011746,307,370,7.101582,0,5,...,4.443579,4.44048,4.443543,0,0,4.443543,4.440374,4.440127,8,94515


In [11]:
test_df[cols_to_norm] = scaler.fit_transform(test_df[cols_to_norm])
test_df['features'] = test_df[cols_to_norm].values.tolist()
test_df

,src,dst,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_referrer,http_version,http_request_body_len,http_response_body_len,http_status_code,http_orig_mime_types,http_resp_mime_types,label,time_group,features
33795,127.0.0.1:42100,127.0.0.1:7878,-0.056165,-0.440364,-0.254414,-0.051715,-0.025944,0.576990,-0.007998,-0.003175,...,2.664535e-15,0.017799,-0.009629,-0.009744,0.000387,-0.009641,-0.016781,5,7134,"[-0.009743866844837315, -0.010045017033470117,..."
33796,192.168.1.190:43530,192.168.1.190:7878,-0.056165,-0.440364,-0.254414,-0.051715,-0.025944,0.576990,-0.007998,-0.003175,...,2.664535e-15,0.017799,-0.009629,-0.009744,0.000387,-0.009641,-0.016781,5,7134,"[-0.009743866844837315, -0.010045017033470117,..."
33797,192.168.1.152:50148,192.168.1.190:53,0.414176,-0.079200,-0.254402,-0.051715,-0.025936,-1.425559,-0.007998,-0.003175,...,2.664535e-15,0.017799,-0.009629,-0.009744,0.000387,-0.009641,-0.016781,5,7134,"[-0.009743866844837315, -0.004117560613920211,..."
33798,192.168.1.152:50148,192.168.1.190:53,0.414176,-0.079200,-0.254413,-0.051706,-0.025944,1.388720,-0.007998,-0.002337,...,2.664535e-15,0.017799,-0.009629,-0.009744,0.000387,-0.009641,-0.016781,5,7134,"[-0.009743866844837315, -0.010045017033470117,..."
33799,192.168.1.152:50148,192.168.1.190:53,0.414176,-0.079200,-0.254413,-0.051706,-0.025944,1.388720,-0.007998,-0.002337,...,2.664535e-15,0.017799,-0.009629,-0.009744,0.000387,-0.009641,-0.016781,5,7134,"[-0.009743866844837315, -0.010045017033470117,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3909020,192.168.1.31:33242,192.168.1.169:443,-0.056165,-3.301662,-0.253593,-0.051641,-0.025897,1.166903,-0.007998,-0.000243,...,2.664535e-15,0.017799,-0.009629,-0.009744,0.000387,-0.009641,-0.016781,8,94515,"[-0.009743866844837315, 0.007737352225179603, ..."
3909021,192.168.1.31:33200,192.168.1.169:443,-0.056165,-3.301662,-0.253591,-0.051639,-0.025897,1.166903,-0.007998,-0.000243,...,2.664535e-15,0.017799,-0.009629,-0.009744,0.000387,-0.009641,-0.016781,8,94515,"[-0.009743866844837315, 0.007737352225179603, ..."
3909022,192.168.1.31:33108,192.168.1.169:443,-0.056165,-3.301662,-0.253555,-0.051641,-0.025897,1.166903,-0.007998,-0.000243,...,2.664535e-15,0.017799,-0.009629,-0.009744,0.000387,-0.009641,-0.016781,8,94515,"[-0.009743866844837315, 0.007737352225179603, ..."
3909023,192.168.1.31:36072,192.168.1.169:5800,-0.056165,2.096842,-0.254097,-0.051692,-0.025934,1.166903,-0.007998,-0.001081,...,2.664535e-15,0.017799,-0.009629,-0.009744,0.000387,-0.009641,-0.016781,8,94515,"[-0.009743866844837315, -0.0011538324041452572..."


In [13]:
X_train = train_df[['src', 'dst', 'features', 'time_group']]
y_train = train_df['label']
X_test = test_df[['src', 'dst', 'features', 'time_group']]
y_test = test_df['label']

X_train.to_csv(Path(DATA_FOLDER) / 'small' / 'X_train.csv', index=False)
y_train.to_csv(Path(DATA_FOLDER) / 'small' / 'y_train.csv', index=False)
X_test.to_csv(Path(DATA_FOLDER) / 'small' / 'X_test.csv', index=False)
y_test.to_csv(Path(DATA_FOLDER) / 'small' / 'y_test.csv', index=False)

idx_label_df = pd.DataFrame(le.classes_)
idx_label_df.columns = ['label']
idx_label_df.to_csv(Path(DATA_FOLDER) / 'small' / 'idx_label.csv', index=True)